In [2]:
# Core Spark SQL for loading and joining the Parquet files
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [3]:
# initializing the spark session
spark = SparkSession.builder \
                    .appName("ML_model") \
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/04 04:09:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# 1. Reading the data

In [4]:
from pyspark.sql.functions import col, sum as _sum, when

# 1. Define the base path to your cleaned Parquet directories
base_path = "file:///home/ahmedhelmy/Collage/Samsung/Project_1/Dataset/cleaned_data"

# 2. Load the Parquet data into DataFrames
orders_df = spark.read.parquet(f"{base_path}/orders_clean")
items_df = spark.read.parquet(f"{base_path}/order_items_clean")
customers_df = spark.read.parquet(f"{base_path}/customers_clean")
products_df = spark.read.parquet(f"{base_path}/products_clean")

In [5]:
# 3. Aggregate Item-Level Data to Order-Level
# Join items with products to get weights, then sum them up per order
items_with_products = items_df.join(products_df, on="product_id", how="left")

order_features = items_with_products.groupBy("order_id").agg(
    _sum("freight_value").alias("total_freight"),
    _sum("product_weight_g").alias("total_weight_g")
)

In [6]:
# 4. Build the Master DataFrame (Join Orders, Customers, and Features)
master_df = orders_df \
    .join(customers_df, on="customer_id", how="left") \
    .join(order_features, on="order_id", how="left")

# 2. Engineer the target (is_late) column

In [7]:
from pyspark.sql.functions import datediff, dayofweek, col, when

# 1. Engineer the target label (from earlier)
ml_df = master_df.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])
ml_df = ml_df.withColumn(
    "is_late",
    when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0)
)

# 2. Engineer the NEW temporal features
ml_df = ml_df.withColumn(
    "approval_delay_days", 
    datediff(col("order_approved_at"), col("order_purchase_timestamp"))
)
ml_df = ml_df.withColumn(
    "purchase_day_of_week", 
    dayofweek(col("order_purchase_timestamp"))
)
ml_df = ml_df.na.fill(0, subset=["approval_delay_days", "purchase_day_of_week"])

# 3. UPDATED SELECT STATEMENT: Keep the new columns!
final_ml_df = ml_df.select(
    "order_id",
    "customer_state",
    "total_freight",
    "total_weight_g",
    "approval_delay_days",    # <-- New feature included
    "purchase_day_of_week",   # <-- New feature included
    "is_late"
)

# 3. Indexing and Vectorizing Features

In [8]:
# 1. Convert string categories into numerical indices
indexer = StringIndexer(inputCol="customer_state", outputCol="customer_state_index", handleInvalid="keep")
indexed_df = indexer.fit(final_ml_df).transform(final_ml_df)

In [9]:
# 2. Fill any hidden nulls in the numerical columns
clean_ml_df = indexed_df.na.fill(0, subset=["total_freight", "total_weight_g"])

In [10]:
from pyspark.sql.functions import datediff, dayofweek

# 1. Calculate 'Approval Delay' in days
ml_df = ml_df.withColumn(
    "approval_delay_days", 
    datediff(col("order_approved_at"), col("order_purchase_timestamp"))
)

# 2. Extract the Day of the Week (1 = Sunday, 7 = Saturday)
ml_df = ml_df.withColumn(
    "purchase_day_of_week", 
    dayofweek(col("order_purchase_timestamp"))
)

# Clean up any nulls that might have been introduced by missing approval dates
ml_df = ml_df.na.fill(0, subset=["approval_delay_days", "purchase_day_of_week"])

In [11]:
# 3. Pack the individual columns into a single 'features' vector
assembler = VectorAssembler(
    inputCols=["customer_state_index", "total_freight", "total_weight_g", "approval_delay_days", "purchase_day_of_week"],
    outputCol="features"
)

In [12]:
assembled_df = assembler.transform(clean_ml_df)

In [13]:
assembled_df.select("order_id", "customer_state", "features", "is_late").show(5, truncate=False)

+--------------------------------+--------------+--------------------------+-------+
|order_id                        |customer_state|features                  |is_late|
+--------------------------------+--------------+--------------------------+-------+
|0032d07457ae9c806c79368d7d9ce96b|RJ            |[1.0,27.19,3150.0,0.0,7.0]|1      |
|0045e3085f083f0f38d24bb3f22e6593|SP            |[0.0,13.84,1350.0,1.0,5.0]|0      |
|00975e709cd12c34d3bd44e4a7f7a876|SP            |[0.0,7.86,422.0,0.0,2.0]  |0      |
|00995d799817ecc3bd2abd8fbe59c430|RJ            |[1.0,12.92,2550.0,0.0,5.0]|0      |
|00c826c901ab01b6f598a8f0c9b56b82|SP            |[0.0,13.91,7900.0,0.0,6.0]|0      |
+--------------------------------+--------------+--------------------------+-------+
only showing top 5 rows



# 4. Random Forest Training and Evaluation

In [14]:
# Check how imbalanced the data is
late_count = assembled_df.filter(col("is_late") == 1).count()
on_time_count = assembled_df.filter(col("is_late") == 0).count()
print(f"Late: {late_count}, On Time: {on_time_count}")

# Calculate the ratio to balance the dataset
ratio = late_count / on_time_count

# Downsample the majority class (On Time) to match the minority class (Late)
balanced_df = assembled_df.sampleBy("is_late", fractions={0: ratio, 1: 1.0}, seed=42)

# Now, split this BALANCED dataframe into train and test
train_data, test_data = balanced_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Records: {train_data.count()}")
print(f"Testing Records: {test_data.count()}")

Late: 7827, On Time: 88649


Training Records: 12639


[Stage 42:======================================>                   (4 + 2) / 6]

Testing Records: 3034


In [15]:
# 2. Define the Random Forest Classifier
# numTrees and maxDepth are hyperparameters you can tweak later to improve accuracy
rf = RandomForestClassifier(
    labelCol="is_late", 
    featuresCol="features", 
    numTrees=150, 
    maxDepth=10, 
    seed=42
)

In [16]:
# 3. Train the Model on the Training Data
print("Training the Random Forest model... This might take a few seconds.")
rf_model = rf.fit(train_data)

Training the Random Forest model... This might take a few seconds.


26/09/04 04:10:35 WARN DAGScheduler: Broadcasting large task binary with size 1556.3 KiB
26/09/04 04:10:39 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/09/04 04:10:43 WARN DAGScheduler: Broadcasting large task binary with size 4.4 MiB
26/09/04 04:10:49 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB
26/09/04 04:10:54 WARN DAGScheduler: Broadcasting large task binary with size 1475.8 KiB
26/09/04 04:10:57 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/09/04 04:11:02 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
                                                                                

In [17]:
# 4. Make Predictions on the unseen Test Data
predictions = rf_model.transform(test_data)

In [18]:
# 5. Evaluate the Model
# We use AUC (Area Under the ROC Curve) as our primary metric.
# An AUC of 0.5 is guessing; 1.0 is perfect prediction.
evaluator = BinaryClassificationEvaluator(
    labelCol="is_late", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

In [19]:
auc = evaluator.evaluate(predictions)
print(f"=====================================")
print(f"Model AUC (Area Under ROC): {auc:.4f}")
print(f"=====================================")

# Show the actual vs predicted results for a few test records
predictions.select("order_id", "is_late", "prediction", "probability").show(5, truncate=False)

26/09/04 04:11:17 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
                                                                                

Model AUC (Area Under ROC): 0.6205


26/09/04 04:11:25 WARN DAGScheduler: Broadcasting large task binary with size 7.7 MiB
[Stage 146:>                                                        (0 + 1) / 1]

+--------------------------------+-------+----------+----------------------------------------+
|order_id                        |is_late|prediction|probability                             |
+--------------------------------+-------+----------+----------------------------------------+
|002b4e6fa42cd4a22cc86abc18fe9c05|0      |0.0       |[0.7364380944081169,0.26356190559188303]|
|003a94f778ef8cfd50247c8c1b582257|1      |1.0       |[0.4080725038637272,0.5919274961362728] |
|004157daafa0bc8672e01f00e4f0c04f|0      |1.0       |[0.4098956652745541,0.5901043347254459] |
|0096668e5b0b8e9657a6f7209a4e58b4|1      |0.0       |[0.5885885121616039,0.4114114878383962] |
|01015fb6493ea5b9ba105c80b5452da1|1      |1.0       |[0.4842589591289239,0.5157410408710762] |
+--------------------------------+-------+----------+----------------------------------------+
only showing top 5 rows

